# PCA of ancient and modern Eurasians

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmadiepeveen/Evolutionary_Thinking_2026/blob/main/week47/Friday/exercise/pca_eurasia.ipynb)

Today we read a PCA instead of running one. You get the **output** of a principal
components analysis of ~1,000 West Eurasian and ~300 East Eurasian individuals, part of it
ancient DNA, and your job is to work out what the plot is telling you about European and
Siberian population history.

The West Eurasian analysis recreates the PCA in
[Mathieson et al. (2015), *Nature* 528:499–503](https://doi.org/10.1038/nature16152),
*Genome-wide patterns of selection in 230 ancient Eurasians*. You don't have to read the paper,
but we will compare our plot to the one in its Figure 1.


**By the end of this notebook you should be able to:**

1. Explain what each number in the data file is: what one row, one `PC` column and one point in the
   plot represent.
2. Explain what *projection* means and why ancient samples are projected onto modern variation
   rather than used to compute the axes.
3. Read a PCA plot: what the axes mean, what they *don't* mean, and why in this dataset you
   **cannot** judge a PC's importance from how wide the cloud looks.
4. Say which real population-genetic processes put the West Eurasian clusters where they are.
5. Judge for yourself when a higher PC still carries population signal and when it is just noise.
6. Spot a data error by plotting before testing — this dataset contains some.

The 📝 **TODO** cells are for you to fill in; each has a 💡 hint, a check cell that tells you whether you got it right, and a ✅ **Reveal solution** block with the code. The answers to the hand-in questions are not in here — those are yours to write.

**Work through it in order.** Sections 1–4 build the ideas; Sections 5–9 are the hand-in questions.
There is a map at the end of Section 0 showing which section answers which hand-in question.

## 0. Setup

### 0.1 (Optional) Install the packages

A notebook runs on a **kernel**: the Python installation that executes your code. If a package isn't
installed in that Python, you get `ModuleNotFoundError: No module named ...`.

The cell below checks which packages are missing and installs **only those**, into the Python that
runs this notebook, so you don't need to set up conda or environments yourself.

* **Run it** if you are on Google Colab, or if you get a `ModuleNotFoundError` later on.
* **Skip it** if you already made the course environment from `env.yaml`.
* It is safe to run more than once. If nothing is missing it does nothing.
* If an import still fails after installing, restart the kernel (*Kernel → Restart*, or
  *Runtime → Restart session* in Colab) and run from the top again.

In [ ]:
# OPTIONAL: installs any missing packages into the Python that runs this notebook
import importlib.util
import subprocess
import sys

# "name you import" : "name you install with pip"
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}

missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", ", ".join(missing), "(this can take a minute)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    print("Done! All packages are installed.")
else:
    print("All packages are already installed - nothing to do.")

### 0.2 The packages we use

| Package | What it is | What we use it for today |
|---|---|---|
| **pandas** (`pd`) | Tables, called *DataFrames*. Think of an Excel sheet, or an R `data.frame`. | The whole dataset: reading the files, subsetting populations, `merge`, group means. |
| **numpy** (`np`) | Fast arrays of numbers; almost all scientific Python is built on it. | A few sums and square roots. |
| **matplotlib** (`plt`) | The standard plotting library. | Every scatter plot and bar plot. |
| **scipy.stats** | Statistical tests and distributions. | `pearsonr` and `spearmanr` for the correlations in Section 8. |

<details>
<summary><b>🐍 Python refresher: the pandas syntax used in this notebook (open if you're rusty)</b></summary>

```python
df.shape                         # (number of rows, number of columns)
df.head()                        # first 5 rows
df.columns                       # the column names
df["PC1"]                        # one column (a Series)
df[["PC1", "PC2"]]               # two columns (a DataFrame) - note the double brackets
df["POP"].unique()               # the different values in a column
df["POP"].nunique()              # how many different values
df["POP"].value_counts()         # how often each value occurs, most common first

df[df["POP"] == "Finnish"]              # keep the rows where POP is exactly "Finnish"
df[df["POP"].isin(["Finnish", "Czech"])]  # keep the rows where POP is one of several values
df[df["ModAnc"] != "Modern"]            # "!=" means "is not"

df.groupby("POP")["PC1"].mean()         # mean of PC1 within each population
df.groupby("POP")[["PC1", "PC2"]].mean()   # ... for several columns at once
df.sort_values("PC1")                   # sort rows by a column

pd.merge(left, right, on="IND")         # join two tables on a shared column
df["new"] = df["old"].str.replace("a", "b")   # string operations on a whole column
```

A `...` (Ellipsis) in a cell is a **placeholder you need to replace**. The check cells below the
TODOs will tell you if you forgot one.

</details>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def check(condition, hint_if_wrong):
    # Small helper for the check cells: prints a tick, or stops with a hint about what is wrong
    if not condition:
        raise AssertionError("Not quite yet - " + hint_if_wrong)
    print("\u2705 Looks good!")


def check_todo(**variables):
    # Stops with a clear message if a TODO still contains the placeholder `...`
    for name, value in variables.items():
        if value is ...:
            raise RuntimeError(
                f"`{name}` is not ready yet: it still contains the placeholder `...` (Ellipsis). "
                f"Finish the TODO that makes `{name}`, run that cell, then run this cell again.")


print("Imports worked!")

### 0.3 Getting the data

The three data files are plain tab-separated text, ~140 kB in total. The cell below looks for them
in a `data/` folder next to this notebook or one folder up (where they live in the course
repository). If it can't find them — for example on Colab — it downloads them from GitHub, falling
back to the zipped copies if need be. You don't need to change anything here.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

FILES = ["dataWE.txt", "dataEE.txt", "dataWE_LatLong.txt"]
RAW = "https://raw.githubusercontent.com/emmadiepeveen/Evolutionary_Thinking_2026/main/week47/Friday/"


def fetch(name, folder):
    """Download one data file into `folder`, from the plain copy or the zipped one."""
    try:
        urllib.request.urlretrieve(RAW + "data/" + name, folder / name)
        return "data/" + name
    except Exception:
        # fall back to the zip archives that sit one level up in the repository
        with urllib.request.urlopen(RAW + name + ".zip") as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        (folder / name).write_bytes(archive.read(name))   # ignores the __MACOSX entry
        return name + ".zip"


def find_or_download_data():
    for folder in [Path("data"), Path("../data"), Path(".")]:
        if all((folder / name).exists() for name in FILES):
            print("Found the data in:", folder.resolve())
            return folder

    print("Data not found locally - downloading (~140 kB)...")
    folder = Path("data")
    folder.mkdir(exist_ok=True)
    for name in FILES:
        print("  downloaded", name, "from", fetch(name, folder))
    return folder


data_dir = find_or_download_data()

### 0.4 Where the hand-in questions are answered

The hand-in questions from the course page are spread through the notebook, each in the section that
gives you the tools for it:

| Hand-in question | Section |
|---|---|
| 1. PC1 vs PC2 of the West Eurasian data, coloured by `ModAnc` | 3 |
| 2. English, Norwegian, Finnish, Icelandic | 5 |
| 3. Four populations of your choice | 5 |
| 4. PC3 vs PC4 of those four | 6 |
| 5. Combinations of PC5–PC10, and why | 7 |
| 6. Correlate latitude/longitude with PC1 and PC2 | 8 |
| 7. PC1 vs PC2 of the East Eurasian data | 9 |
| 8. PC3 vs PC4 of the East Eurasian data | 9 |
| 9. Which aDNA artefacts make ancient samples unsuitable for computing PCs | 2 and 10 |
| 10. What other data can you analyse with PCA | 10 |

Each one is marked **📝 Hand-in question N**. Everything else is there to make those answerable.

## 1. What a PCA actually did to these data

You met PCA in week 44 on goat genotypes. Here is the same idea in four sentences, because
everything in this notebook depends on it.

**Start from the genotype matrix.** Every individual was genotyped at hundreds of thousands of
**SNPs** (positions in the genome where people differ by a single DNA letter). At each SNP you
store how many copies of one chosen allele the individual carries: `0`, `1` or `2`. So each
individual is one row of a huge table of 0s, 1s and 2s.

**That makes each individual a point in a space with one dimension per SNP.** With 500,000 SNPs,
each person is a point in 500,000-dimensional space. Two people with similar genomes sit close
together in that space, because they have similar genotypes at most SNPs.

**PCA finds the directions in which that cloud of points is most stretched out.** The direction of
greatest spread is **PC1**. The direction of greatest remaining spread, at right angles to PC1,
is **PC2**, then PC3, and so on. These directions are combinations of all the SNPs at once, not
single SNPs.

**The "PC1" and "PC2" columns in your data file are where each individual falls along those
directions.** Projecting a 500,000-dimensional cloud onto its two most-stretched directions is the
most informative flat picture of it you can draw — like choosing the angle to photograph a
sculpture from so you lose as little of its shape as possible.

The important consequence: **the axes have no units and no intrinsic meaning.** Nobody designed
PC1 to be "north–south" or "farmer ancestry". It came out of the data as "the direction along which
these particular individuals differ most". What it *means* is something you work out afterwards, by
looking at which populations sit at its ends. We will do exactly that in Section 4.

### ❓ Question: Check your understanding

1. Two individuals end up at almost the same point in the PC1–PC2 plot. What does that tell you? 
2. The sign of a PC is arbitrary: software could return `PC1` or `-PC1` and both are equally
   correct. What does that mean for how you describe a plot?
3. Suppose you add 200 more Finnish individuals to the dataset and re-run the PCA. Could the axes
   change?

<details>
<summary><b>✅ Reveal answer</b></summary>

1. It tells you their genotypes are similar **in the directions PC1 and PC2 describe** — usually
   read as "similar genome-wide ancestry". It does **not** necessarily mean that they are closely related or
   they come from the same place, or that they are similar in the directions PC3, PC4, ... describe.
   
2. Say "Population A and B sit at **opposite ends** of PC1", not "A has a high PC1". If you
   re-ran the analysis the plot might come out mirrored, and that would mean nothing. Only relative
   positions carry information.
3. Yes. PCA finds the directions of greatest spread **in the sample you give it**. Over-representing
   one population makes the differences involving that population a bigger share of the total
   spread, so an axis can rotate towards it. 

</details>

## 2. Where these numbers came from, and why you're not computing them

### 2.1 smartPCA

The analysis was run with `smartPCA` from the [Eigensoft](https://github.com/DReichLab/EIG) package,
the standard tool for this in human population genetics. 
We are **not** running it today — it needs the full genotype data. 


`smartPCA` reads a parameter file pointing at four inputs:

| Input | Contents |
|---|---|
| `.geno` | The genotype matrix: individuals left to right, SNPs top to bottom. |
| `.snp` | One line per SNP: name, chromosome, genetic position, physical position, reference allele, alternative allele. |
| `.ind` | One line per individual: individual name, sex (`M`/`F`/`U`), population name. |
| *population list* | **The list of populations whose individuals are used to compute the PCs.** |


### 2.2 The population list, and projection

Why not compute the axes from everybody? 

**First off:** there are too many individuals and the computation gets slow.

**But the  real reason:** Ancient samples have systematic problems that modern samples don't:

* **Low coverage and lots of missing data.** Each ancient genome is sequenced shallowly, so many
  genotypes are missing or uncertain, and the pattern of missingness differs between samples.
* **Post-mortem damage.** Cytosine deaminates to uracil over time, which is read as a C→T change
  (and G→A on the other strand). Older or worse-preserved samples carry more of these errors.
* **Contamination** with modern human DNA from excavators and lab staff, and with environmental
  microbial DNA.
* **Reference bias.** Short, damaged reads map more readily to the reference allele, so
  heterozygotes get under-called and the direction of the error depends on read length.
* **Batch effects.** Different labs, extraction protocols, capture panels and sequencing runs leave
  their own fingerprints.

These artefacts are **shared by samples processed similarly** and **not shared with modern samples**. They are a large, consistent source of variation.  A leading PC can easily end up describing *sequencing batch* or *how degraded the sample is* rather than ancestry.
You would get a beautiful plot of your laboratory's history.

**The fix is projection.** The PCs are computed from the modern populations in the population list
only. Then **every** individual, ancient ones included, is placed on those already-fixed axes by
taking their genotypes and working out where they fall along each direction. 

This is a trade-off:  Ancient populations may carry variation that no modern population has, and projection is blind to it by construction: an axis that was never drawn cannot show you anything. Projection also shrinks projected points towards the origin — the
more missing data a sample has, the more it shrinks — so an ancient individual sitting between two
modern clusters is not automatically admixed between them.
In practice, avoiding the artefacts is worth more than the lost variation, and the approach breaks down only once the samples get so old
and so distant from modern variation that the modern axes are no longer informative about them.

## 3. The West Eurasian data

### 3.1 Reading the file

`dataWE.txt` is tab-separated with a header line, so `sep="\t"` and the default `header=0`.

In [ ]:
dataWE = pd.read_csv(data_dir / "dataWE.txt", sep="\t")

print("rows (individuals), columns:", dataWE.shape)
dataWE.head()

Each **row is one individual**. The columns are:

| Column | Meaning |
|---|---|
| `IND` | The individual's ID. Modern individuals have IDs like `abh24.HO`; ancient ones like `RISE507` or `I0061`. |
| `PC1` … `PC10` | Where that individual falls along each of the first ten principal components. |
| `POP` | The population the individual is assigned to. |
| `ModAnc` | A grouping of the populations — see below, it is not what you'd expect. |

### 📝 TODO: Get a feel for the dataset

Work out:

1. how many individuals there are in total (`n_individuals`),
2. how many different populations (`n_populations`),
3. which populations have the most individuals (show the top 10).

<details>
<summary><b>💡 Hint (click to open)</b></summary>

* `len(df)` is the number of rows; `df.shape` gives (rows, columns).
* `df["col"].nunique()` counts how many *different* values a column has.
* `df["col"].value_counts()` counts how often each value occurs, most common first. Chain `.head(10)`
  to keep the first ten.

</details>

In [ ]:
n_individuals = ...                 # <- your code
n_populations = ...                 # <- your code

print("Individuals:", n_individuals)
print("Populations:", n_populations)

...                                 # <- your code: the 10 populations with the most individuals

In [ ]:
check_todo(n_individuals=n_individuals, n_populations=n_populations)
check(n_individuals == 997, "n_individuals should be the number of rows of dataWE.")
check(n_populations == 89, "n_populations should count the *different* values in the POP column.")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
n_individuals = len(dataWE)
n_populations = dataWE["POP"].nunique()

print("Individuals:", n_individuals)
print("Populations:", n_populations)

print(dataWE["POP"].value_counts().head(10))
```

</details>

### 3.2 The `ModAnc` column 

In [ ]:
dataWE["ModAnc"].value_counts()

They are **archaeological/ancestry groupings**:

| `ModAnc` | Who | Roughly when |
|---|---|---|
| `Modern` | Present-day West Eurasians. **These are the individuals the PCs were computed from.** Everything else is projected. | today |
| `Hunter-gatherer` | Western and Scandinavian Mesolithic hunter-gatherers (`WHG`, `Motala_HG`) | ~9,000–5,000 BCE |
| `EHG` | **E**astern European **h**unter-**g**atherers, from Karelia and the Samara region | ~8,000–5,000 BCE |
| `Early-farmer` | Neolithic farmers of Anatolian origin and their European descendants (`Anatolia_Neolithic`, `LBK_EN`, `Iberia_EN`/`MN`, `Hungary_EN`, `Central_MN`, `Remedello`, `Iberia_Chalcolithic`, and the Iceman) | ~7,000–2,500 BCE |
| `Steppe` | Bronze Age pastoralists of the Pontic–Caspian steppe (`Yamnaya`, `Afanasievo`, `Poltavka`, `Sintashta`, `Andronovo`, `Srubnaya`) **and** the later central/northern European groups that carry steppe ancestry (`Central_LNBA` ≈ Corded Ware, `Bell_Beaker_LN`, `Northern_LNBA`, `Hungary_BA`) | ~3,300–1,000 BCE |
| `Samara_Eneolithic` | Two Copper Age individuals from the Samara region, left as their own category | ~5,000–4,000 BCE |

In [ ]:
# one row per (ModAnc, POP) combination, with the number of individuals
by_group = dataWE.groupby(["ModAnc", "POP"]).size().rename("n")
print(by_group.to_string())

###  Two things in this output are data errors, not biology

Look carefully at the population names and you will spot:

**1. Some names are mangled: `Modernaltese`, `Modernordovian`, `Modernotala_HG`.**
These are `Maltese`, `Mordovian` and `Motala_HG`. At some point somebody replaced the letter `M`
with the word `Modern` across the whole file — presumably turning a `ModAnc` column that said `M`
into one that said `Modern` — and the replacement also hit every population name starting with `M`.
No other letter was affected, which is why `Anatolia_Neolithic` survived intact.

**2. `Anatolia_Neolithic` appears in two `ModAnc` groups** (21 individuals as `Early-farmer`,
3 as `Steppe`), and `Samara_Eneolithic` appears both as its own `ModAnc` and inside `Steppe`.
These are labelling inconsistencies in how the file was assembled. Anatolian Neolithic individuals
are early farmers; the three filed under `Steppe` are mislabelled.

Neither error affects the PC values themselves — the coordinates are fine, only the labels are
wrong. We will fix the names, leave the duplicated `ModAnc` assignment alone (it is three
individuals out of 997), and note it when it matters.

### 📝 TODO: Fix the mangled population names

Make a `POP` column with `Modernaltese` → `Maltese`, `Modernordovian` → `Mordovian`,
`Modernotala_HG` → `Motala_HG`.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

* `df["col"].replace({"old": "new", ...})` swaps **whole values** using a dictionary. (That is what
  you want here. `.str.replace("Modern", "M")` would also hit the word `Modern` itself in the
  `ModAnc` column if you ever applied it there, so whole-value replacement is the safer tool.)
* Assign the result back: `dataWE["POP"] = ...`

</details>

In [ ]:
name_fixes = {
    "Modernaltese": "Maltese",
    "Modernordovian": "Mordovian",
    "Modernotala_HG": "Motala_HG",
}

dataWE["POP"] = ...                 # <- your code

print(sorted(p for p in dataWE["POP"].unique() if p.startswith("M")))

In [ ]:
check("Maltese" in set(dataWE["POP"]), "Maltese should now be one of the POP values.")
check("Modernaltese" not in set(dataWE["POP"]), "Modernaltese should be gone.")
check("Motala_HG" in set(dataWE["POP"]), "Motala_HG should now be one of the POP values.")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
name_fixes = {
    "Modernaltese": "Maltese",
    "Modernordovian": "Mordovian",
    "Modernotala_HG": "Motala_HG",
}

dataWE["POP"] = dataWE["POP"].replace(name_fixes)

print(sorted(p for p in dataWE["POP"].unique() if p.startswith("M")))
```

</details>

## 4. The headline plot

### 4.1 A helper for plotting PCs.

In [ ]:
def plot_pcs(df, x="PC1", y="PC2", colour="POP", ax=None, title=None,
             size=22, alpha=0.85, legend=True, cmap_name=None):
    """Scatter two PC columns of `df`, with one colour per distinct value of the `colour` column.

    Returns the matplotlib Axes, so you can keep customising it afterwards.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 6))

    groups = sorted(df[colour].unique())
    if cmap_name is None:
        cmap_name = "tab10" if len(groups) <= 10 else "tab20"
    cmap = plt.get_cmap(cmap_name)

    for i, group in enumerate(groups):
        rows = df[df[colour] == group]
        ax.scatter(rows[x], rows[y], s=size, alpha=alpha, label=group,
                   color=cmap(i % cmap.N), edgecolor="none")

    ax.axhline(0, lw=0.5, color="0.8", zorder=0)
    ax.axvline(0, lw=0.5, color="0.8", zorder=0)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    if title:
        ax.set_title(title)
    if legend and len(groups) <= 25:
        ax.legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
    return ax


print("plot_pcs is defined.")

### 4.2 📝 Hand-in question 1: PC1 vs PC2, coloured by `ModAnc`

Plot the first two principal components of `dataWE`, with one colour per `ModAnc` group.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

One line: `plot_pcs(dataWE, colour="ModAnc", title="...")`, then `plt.show()`.

(`x="PC1"` and `y="PC2"` are already the defaults.)

</details>

In [ ]:
...                                 # <- your code: PC1 vs PC2, coloured by ModAnc
plt.show()

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
plot_pcs(dataWE, x="PC1", y="PC2", colour="ModAnc",
         title="West Eurasian PCA: modern individuals and projected ancient groups")
plt.show()
```

</details>

### 4.3 Reading the plot

You should see a large arc or crescent of `Modern` individuals, with the ancient groups falling
inside and along it. This is the structure in Figure 1 of Mathieson et al. (2015), and the reason
it is such a famous plot is that **the ancient groups land in interpretable places**.

To interpret the axes: look at which populations sit at the ends.
Because the modern individuals define the axes, we look at *their* extremes.

In [ ]:
PCS = [f"PC{i}" for i in range(1, 11)]

modern = dataWE[dataWE["ModAnc"] == "Modern"]
pop_means = modern.groupby("POP")[PCS].mean()

for pc in ["PC1", "PC2"]:
    ordered = pop_means[pc].sort_values()
    low = ", ".join(f"{p} ({v:+.3f})" for p, v in ordered.head(5).items())
    high = ", ".join(f"{p} ({v:+.3f})" for p, v in ordered.tail(5).items())
    print(f"{pc}\n   one end: {low}\n   other end: {high}\n")

That gives the axes their meaning:

* **PC1** runs from the **northeast European** populations (Lithuanian, Estonian, Finnish, Russian)
  at one end to the **Arabian peninsula and Near East** (Bedouin, Yemenite Jews, Saudi) at the
  other. It is essentially the southeast–northwest cline of West Eurasian variation.
* **PC2** separates the **Caucasus** (Abkhasian, North Ossetian, Chechen, Georgian, Lezgin) from
  **Sardinians, Basques and Bedouins**. Combined with PC1 this spreads the modern populations into
  the familiar arc, with Europe along one arm and the Near East along the other.

Now the ancient groups:

In [ ]:
ancient = dataWE[dataWE["ModAnc"] != "Modern"]
print(ancient.groupby("ModAnc")[["PC1", "PC2"]].mean().round(4).to_string())
print("\nindividuals per ancient group:")
print(ancient["ModAnc"].value_counts().to_string())

In [ ]:
# A few modern populations to orient yourself by. Change these if you like.
reference_pops = ["Sardinian", "Basque", "Spanish", "French", "English", "Norwegian",
                  "Finnish", "Estonian", "Lithuanian", "Russian", "Greek", "Turkish",
                  "Abkhasian", "Druze", "BedouinA", "BedouinB"]

ancient_colours = {"Hunter-gatherer": "tab:blue", "EHG": "tab:cyan",
                   "Early-farmer": "tab:green", "Steppe": "tab:red",
                   "Samara_Eneolithic": "tab:purple"}

ancient_means = ancient.groupby("ModAnc")[["PC1", "PC2"]].mean()

fig, ax = plt.subplots(figsize=(10, 8))

# every modern individual
ax.scatter(modern["PC1"], modern["PC2"], s=14, color="0.88")

# the modern populations you are orienting by
for pop in reference_pops:
    x, y = pop_means.loc[pop, "PC1"], pop_means.loc[pop, "PC2"]
    ax.annotate(pop, (x, y))

# the ancient individuals, faintly
for group, colour in ancient_colours.items():
    rows = ancient[ancient["ModAnc"] == group]
    ax.scatter(rows["PC1"], rows["PC2"], color=colour)

# the ancient GROUP MEANS, which is what the question is about
for group, colour in ancient_colours.items():
    x, y = ancient_means.loc[group, "PC1"], ancient_means.loc[group, "PC2"]
    ax.scatter([x], [y], color=colour, s =260, edgecolors="black")
    ax.annotate(group, (x, y))

ax.axhline(0, lw=0.5, color="0.9", zorder=0)
ax.axvline(0, lw=0.5, color="0.9", zorder=0)
ax.set_xlabel("PC1   (northeast Europe \u2190\u2192 Near East)")
ax.set_ylabel("PC2   (Sardinia / Basque / Bedouin \u2190\u2192 Caucasus)")
ax.set_title("Where each ancient group sits (diamonds = group means)\n"
             "grey = modern individuals, pale colour = individual ancient samples")
plt.tight_layout()
plt.show()

### ❓ Question: What the ancient groups' positions mean

Given that PC1 runs northeast-Europe (negative) to Near-East (positive), and looking at where each
ancient group sits, work out the story. Think about it before revealing.

1. `Hunter-gatherer` and `EHG` sit at the far **negative** end of PC1, beyond every modern
   population. What does "beyond the modern range" tell you?
2. `Early-farmer` sits near PC1 ≈ 0 with a clearly **negative PC2**, close to modern Sardinians.
   Why Sardinians?
3. `Steppe` sits **between** the hunter-gatherers and the early farmers. What is the standard
   interpretation, and what is the caveat from Section 2?
4. Modern northern Europeans sit between all three ancient groups rather than on top of any one of
   them. What does that suggest?

<details>
<summary><b>✅ Reveal answer</b></summary>

1. No present-day population is a direct, unmixed continuation of European Mesolithic
   hunter-gatherers. Every modern European carries substantial later ancestry (farmer, steppe), which
   pulls them away from the hunter-gatherer extreme. That ancestry has not disappeared — it is
   diluted. (Part of the distance is also projection shrinkage and the low coverage of these few
   samples, so don't over-read the exact position of a 3-individual group.)
2. Sardinians are the modern population with the **highest proportion of early-farmer ancestry**,
   because Sardinia received comparatively little of the later steppe-related gene flow that reshaped
   mainland Europe. They are often described as the closest modern proxy for Neolithic European
   farmers — which is why early farmers project right next to them.
3. The standard interpretation is that Bronze Age steppe populations were themselves a mixture, with
   roughly half their ancestry from eastern hunter-gatherers and much of the rest from Caucasus and
   Near-Eastern-related sources — and that their expansion into Europe after ~3000 BCE brought that
   ancestry west. The caveat: *this plot alone* does not establish any of it. An intermediate
   position is consistent with admixture but also with shared drift or with projection shrinkage.
   The admixture conclusion comes from f-statistics and qpAdm in the original papers, not from the
   scatter plot.
4. That modern northern Europeans are **mixtures of all three** ancient sources — hunter-gatherer,
   early farmer and steppe — in varying proportions. 

</details>

### 4.4 The modern populations, for orientation

The plot above named sixteen populations to orient you. Here is the same space with every
modern population labelled at its mean, and the ancient groups left off so that nothing is
hidden behind them. Read it as a map: the arc really does run from the Baltic, round through
central and southern Europe, to the Levant and the Arabian peninsula. Keep it to hand — Sections
6 and 7 ask you to interpret small groups of these populations. (Nothing to do here.)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

# every modern individual, as a pale backdrop
ax.scatter(modern["PC1"], modern["PC2"], s=12, color="0.85", edgecolor="none", zorder=1)

# One label per population, at its mean. matplotlib has no "don't draw colliding
# labels" option, so we do it ourselves: go through the populations and skip any
# whose label would land on top of one we have already placed.
min_gap = 0.004                     # in PC units; raise it for fewer, clearer labels
placed = []
for pop, row in pop_means.sort_values("PC1").iterrows():
    x, y = row["PC1"], row["PC2"]
    if any(abs(x - px) < min_gap and abs(y - py) < min_gap for px, py in placed):
        continue
    placed.append((x, y))
    ax.annotate(pop, (x, y), fontsize=6.5, color="0.35",
                ha="center", va="center", zorder=2)

print(f"{len(placed)} of {len(pop_means)} population labels drawn "
      f"(the rest would have overlapped)")

ax.axhline(0, lw=0.5, color="0.9", zorder=0)
ax.axvline(0, lw=0.5, color="0.9", zorder=0)
ax.set_xlabel("PC1   (northeast Europe \u2190\u2192 Near East)")
ax.set_ylabel("PC2   (Sardinia / Basque / Bedouin \u2190\u2192 Caucasus)")
ax.set_title("A map of modern West Eurasian variation\n"
             "every modern individual in grey, each population labelled at its mean")
plt.tight_layout()
plt.show()

## 5. Warning
`smartPCA` writes the eigenvalues to a separate output file, and you were only given the coordinates. So you cannot report variance explained for
these PCs.


In [ ]:
# For each PC: the sum of squared scores, over the modern individuals only
sumsq = (modern[PCS] ** 2).sum()
print("sum of squared scores, modern individuals:")
print(sumsq.round(4).to_string())

print("\nstandard deviation, modern individuals:")
print(modern[PCS].std().round(4).to_string())

print(f"\n1 / sqrt(n - 1) with n = {len(modern)}: {1 / np.sqrt(len(modern) - 1):.4f}")

Every PC has sum of squares **exactly 1**, and therefore exactly the same standard deviation,
0.0359, which is precisely $1/\sqrt{n-1}$ for the 777 modern individuals.

`smartPCA` **normalised every principal component to unit length.** The eigenvalue — the thing that
says how important a PC is — has been divided out of the coordinates and stored elsewhere. So:

> **In this dataset, PC10 occupies exactly as much width on the page as PC1.**
> The spread of the cloud carries **no** information about how much variance a PC explains.


A couple of smaller consequences worth knowing:

* The spread of **all 997** individuals is slightly wider than 0.0359 on PC1–PC4. That is the
  projected ancient individuals, who were not part of the normalisation. On PC5–PC10 it is back to
  0.0318 — the ancients barely budge from the origin on those axes, which is itself a hint about
  how much population signal is left in them.
* The East Eurasian file in Section 9 was **not** normalised the same way (its sums of squares are
  not 1 and differ between PCs), so don't assume the two files are on a common scale. They are
  separate analyses with separate axes; a `PC1` in one has nothing to do with a `PC1` in the other.

In [ ]:
print("all 997 individuals, sd per PC:")
print(dataWE[PCS].std().round(4).to_string())

## 6. Zooming in on a few populations

### 6.1 📝 Hand-in question 2: the North Atlantic and the Baltic

Subset `dataWE` to the **English, Norwegian, Finnish and Icelandic** populations and plot PC1 vs
PC2, with one colour per population.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

* `df[df["POP"].isin(["English", ...])]` keeps the rows whose `POP` is one of a list of values.
* Then `plot_pcs(subset, colour="POP", title="...")`.

</details>

In [ ]:
nordic_pops = ["English", "Norwegian", "Finnish", "Icelandic"]

nordic = ...                        # <- your code: the rows of dataWE in those four populations

print(nordic["POP"].value_counts().to_string())
...                                 # <- your code: plot PC1 vs PC2, coloured by POP
plt.show()

In [ ]:
check_todo(nordic=nordic)
check(len(nordic) == 40, "nordic should have 40 rows: 12 Icelandic + 11 Norwegian + 10 English + 7 Finnish.")
check(set(nordic["POP"]) == set(nordic_pops), "nordic should contain exactly those four populations.")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
nordic_pops = ["English", "Norwegian", "Finnish", "Icelandic"]

nordic = dataWE[dataWE["POP"].isin(nordic_pops)]

print(nordic["POP"].value_counts().to_string())
plot_pcs(nordic, x="PC1", y="PC2", colour="POP",
         title="PC1 vs PC2: English, Norwegian, Finnish, Icelandic")
plt.show()
```

</details>

In [ ]:
print("population means:")
print(nordic.groupby("POP")[["PC1", "PC2", "PC3", "PC4"]].mean().round(4).to_string())
print("\nstandard deviation within each population:")
print(nordic.groupby("POP")[["PC1", "PC2", "PC3", "PC4"]].std().round(4).to_string())

### ❓ Question: Interpret the four populations on PC1 and PC2

Which populations overlap, which separate, is that what geography would predict,
and how much should you trust it given the spread within each population?


### 6.2 📝 Hand-in question 3: four populations of your choice

Now pick your own four and do the same. Some suggestions, each of which shows something:

* `["Sardinian", "Italian_North", "Italian_South", "Greek"]` — a cline up the Italian peninsula, and
  Sardinia's isolation.
* `["Basque", "Spanish", "Spanish_North", "French"]` — a population isolate next to its neighbours.
* `["Druze", "Palestinian", "Lebanese", "Syrian"]` — Levantine populations, one of which is an
  endogamous religious community.
* `["Jew_Ashkenazi", "Jew_Moroccan", "Jew_Yemenite", "Jew_Turkish"]` — one diaspora, several host
  regions. Do they cluster with each other or with where they lived?
* `["Finnish", "Russian", "Estonian", "Lithuanian"]` — the northeastern end of PC1 in detail.
* Or mix moderns and ancients: `["Sardinian", "Anatolia_Neolithic", "LBK_EN", "WHG"]`.

Print the full list of populations first, pick four, plot them, and write down what you see.

In [ ]:
print(sorted(dataWE["POP"].unique()))

In [ ]:
my_pops = ...                       # <- your code: a list of four population names

my_subset = ...                     # <- your code
...                                 # <- your code: plot PC1 vs PC2, coloured by POP
plt.show()

print(my_subset.groupby("POP")[["PC1", "PC2"]].agg(["mean", "std"]).round(4).to_string())

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
my_pops = ["Sardinian", "Italian_North", "Italian_South", "Greek"]

my_subset = dataWE[dataWE["POP"].isin(my_pops)]
plot_pcs(my_subset, x="PC1", y="PC2", colour="POP", title="PC1 vs PC2: " + ", ".join(my_pops))
plt.show()

print(my_subset.groupby("POP")[["PC1", "PC2"]].agg(["mean", "std"]).round(4).to_string())
```

</details>

## 7. Higher principal components

### 7.1 📝 Hand-in question 4: PC3 vs PC4 of your four Nordic populations

Plot PC3 against PC4 for the English/Norwegian/Finnish/Icelandic subset. Compare with the PC1–PC2
plot: has the relationship between the populations changed?

<details>
<summary><b>💡 Hint (click to open)</b></summary>

Same call as before with `x="PC3", y="PC4"`. To compare properly, draw them side by side:

```python
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_pcs(nordic, x="PC1", y="PC2", colour="POP", ax=axes[0], legend=False)
plot_pcs(nordic, x="PC3", y="PC4", colour="POP", ax=axes[1])
```

</details>

In [ ]:
...                                 # <- your code: PC1-PC2 and PC3-PC4 side by side
plt.show()

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_pcs(nordic, x="PC1", y="PC2", colour="POP", ax=axes[0], legend=False, title="PC1 vs PC2")
plot_pcs(nordic, x="PC3", y="PC4", colour="POP", ax=axes[1], title="PC3 vs PC4")
plt.tight_layout()
plt.show()
```

</details>

The plot shows the populations sitting further apart on some axes than others. These are the
numbers behind that: the population means on PC1–PC4, and Cohen’s d for the comparisons
that matter — the distance between two group means in units of their pooled standard deviation.
Roughly, d = 1 means the means are one standard deviation apart, and by d ≈ 3 the two groups
barely overlap.

In [ ]:

def cohens_d(a, b, pc):
    """Distance between two group means on one PC, in units of their pooled standard deviation."""
    n_a, n_b = len(a), len(b)
    pooled_sd = np.sqrt(((n_a - 1) * a[pc].var() + (n_b - 1) * b[pc].var()) / (n_a + n_b - 2))
    return (a[pc].mean() - b[pc].mean()) / pooled_sd


def ranges_overlap(a, b, pc):
    """Do the two groups' ranges overlap at all on this axis?"""
    return max(a[pc].min(), b[pc].min()) <= min(a[pc].max(), b[pc].max())


pcs4 = ["PC1", "PC2", "PC3", "PC4"]

finnish = nordic[nordic["POP"] == "Finnish"]
other_three = nordic[nordic["POP"] != "Finnish"]
english = nordic[nordic["POP"] == "English"]
nor_ice = nordic[nordic["POP"].isin(["Norwegian", "Icelandic"])]
norwegian = nordic[nordic["POP"] == "Norwegian"]
icelandic = nordic[nordic["POP"] == "Icelandic"]

print("population means:")
print(nordic.groupby("POP")[pcs4].mean().round(4).to_string())

separation = pd.DataFrame({
    "PC": pcs4,
    "finn_d": [cohens_d(finnish, other_three, pc) for pc in pcs4],
    "finn_overlap": [ranges_overlap(finnish, other_three, pc) for pc in pcs4],
    "english_d": [cohens_d(english, nor_ice, pc) for pc in pcs4],
    "norway_vs_iceland": [cohens_d(norwegian, icelandic, pc) for pc in pcs4],
})

print("\nseparation on each axis (Cohen's d):")
print(separation.round(2).to_string(index=False))

### ❓ Question: Has the relationship changed?


### 7.2 📝 Hand-in question 5: PC5 to PC10

Plot some combinations of PC5–PC10 for the same four populations. What happens to the separation
between populations, and why?

Rather than plotting one pair at a time, let's plot all five consecutive pairs at once. This is the
exercise where you use `plot_pcs` in a loop.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

```python
pairs = [("PC5", "PC6"), ("PC6", "PC7"), ...]
fig, axes = plt.subplots(1, len(pairs), figsize=(4.2 * len(pairs), 4.2))
for ax, (x, y) in zip(axes, pairs):
    plot_pcs(nordic, x=x, y=y, colour="POP", ax=ax, legend=False)
axes[-1].legend(...)
```

To compare fairly across panels, give every panel the same axis limits — otherwise each one
auto-zooms to its own data and they all look equally spread out. `ax.set_xlim(-0.02, 0.02)` and
`ax.set_ylim(-0.02, 0.02)` work well here.

</details>

In [ ]:
pairs = ...                         # <- your code: a list of (x, y) PC name pairs from PC5-PC10

...                                 # <- your code: one panel per pair, shared axis limits
plt.show()

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
pairs = [("PC5", "PC6"), ("PC6", "PC7"), ("PC7", "PC8"), ("PC8", "PC9"), ("PC9", "PC10")]

fig, axes = plt.subplots(1, len(pairs), figsize=(4.0 * len(pairs), 4.2), sharex=True, sharey=True)
for ax, (x, y) in zip(axes, pairs):
    plot_pcs(nordic, x=x, y=y, colour="POP", ax=ax, legend=False, title=f"{x} vs {y}")
    ax.set_xlim(-0.022, 0.022)
    ax.set_ylim(-0.022, 0.022)
axes[-1].legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()
```

</details>

### 7.3 Quantifying "separation"

The natural measure: of all the variation along a given PC, **what share lies between populations
rather than within them?** If populations are cleanly separated, most of the spread is between
group means. If the labels are meaningless on that axis, almost all the spread is within groups.

That quantity is called $\eta^2$ (eta squared), and it is a close relative of $F_{ST}$ — which you
have met as the share of genetic variation that lies between populations rather than within them.
Same idea, applied to one PC instead of to allele frequencies.

$$\eta^2 = \frac{\sum_g n_g (\bar{x}_g - \bar{x})^2}{\sum_i (x_i - \bar{x})^2}
= \frac{\text{variation between population means}}{\text{total variation}}$$

It runs from 0 (the population labels tell you nothing about where an individual falls on this axis)
to 1 (knowing the population tells you the position exactly).

In [ ]:
def eta_squared(df, value, group):
    """Share of the variation in `value` that lies *between* groups. 0 = none, 1 = all."""
    grand_mean = df[value].mean()
    ss_total = ((df[value] - grand_mean) ** 2).sum()
    ss_between = sum(len(rows) * (rows[value].mean() - grand_mean) ** 2
                     for _, rows in df.groupby(group))
    return ss_between / ss_total


# how well does each PC separate the four Nordic populations?
separation = pd.DataFrame({
    "eta_squared": [eta_squared(nordic, pc, "POP") for pc in PCS],
}, index=PCS)

# a formal test alongside it: does any population mean differ on this PC?
separation["p_value"] = [
    stats.f_oneway(*[rows[pc].values for _, rows in nordic.groupby("POP")]).pvalue
    for pc in PCS
]

print(separation.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colours = ["tab:green" if p < 0.01 else "tab:grey" for p in separation["p_value"]]
ax.bar(separation.index, separation["eta_squared"], color=colours)
ax.set_ylabel("$\\eta^2$: share of variation between populations")
ax.set_title("How well does each PC separate English / Norwegian / Finnish / Icelandic?\n"
             "(green = population means differ significantly, p < 0.01)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### ❓ Question: Explain the pattern in that bar plot

What does $\eta^2$ do across PC1 → PC10, and *why*?


In [ ]:
for pc in ["PC5", "PC6", "PC7", "PC8", "PC9", "PC10"]:
    ordered = pop_means[pc].sort_values()
    low = ", ".join(f"{p}" for p in ordered.head(3).index)
    high = ", ".join(f"{p}" for p in ordered.tail(3).index)
    print(f"{pc:>5}:  {low}   <-->   {high}")

## 8. Geography against genetics

### 8.1 📝 Hand-in question 6: do the ancient samples' coordinates predict their PCs?

`dataWE_LatLong.txt` gives the latitude and longitude of the site each **ancient** individual was
excavated from. If the principal components track ancestry, and ancestry varies across space, the
coordinates should correlate with the PCs.

Load it and merge it onto `dataWE` by the `IND` column.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

* `pd.read_csv(data_dir / "dataWE_LatLong.txt", sep="\t")`
* `pd.merge(dataWE, latlong, on="IND")` keeps only the rows present in **both** tables (an inner
  join, the default) — which is what you want, since only ancient individuals have coordinates.

</details>

In [ ]:
latlong = ...                       # <- your code: read dataWE_LatLong.txt
geo = ...                           # <- your code: merge dataWE and latlong on IND

print("rows after merging:", len(geo))
print("ancient individuals in dataWE:", (dataWE["ModAnc"] != "Modern").sum())
geo.head()

In [ ]:
check_todo(latlong=latlong, geo=geo)
check(len(latlong) == 191, "dataWE_LatLong.txt has 191 rows.")
check(len(geo) == 191, "The merge should give 191 rows - an inner join on IND.")
check("Latitude" in geo.columns and "PC1" in geo.columns,
      "geo should have the columns of both tables.")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
latlong = pd.read_csv(data_dir / "dataWE_LatLong.txt", sep="\t")
geo = pd.merge(dataWE, latlong, on="IND")

print("rows after merging:", len(geo))
print("ancient individuals in dataWE:", (dataWE["ModAnc"] != "Modern").sum())
geo.head()
```

</details>

**First observation, before any statistics:** the merge gives **191** rows but there are **220**
ancient individuals. 29 ancient individuals have no coordinates, so they drop out silently. This is
normal for ancient DNA — provenance information is not always published — but you should say so when
you report the result, because an inner join quietly changing your sample size is one of the most
common ways to mislead yourself.

In [ ]:
missing_coords = dataWE[(dataWE["ModAnc"] != "Modern") & (~dataWE["IND"].isin(latlong["IND"]))]
print(f"{len(missing_coords)} ancient individuals without coordinates, from these populations:")
print(missing_coords["POP"].value_counts().to_string())

### 8.2  Plotting before testing

 **Plot the coordinates first.** They
are latitudes and longitudes, so you already know what they should look like: a rough map of West
Eurasia.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(geo["Longitude"], geo["Latitude"], s=28, alpha=0.7, color="tab:red", edgecolor="none")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Where the ancient samples come from")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Latitude: ", geo["Latitude"].min(), "to", geo["Latitude"].max())
print("Longitude:", geo["Longitude"].min(), "to", geo["Longitude"].max())

### ❓ Question: What is wrong with that map?

Look at the range of the latitudes, and at the stray points.

<details>
<summary><b>✅ Reveal answer</b></summary>

Latitude runs from **10.81** to **85.45**. Both ends are impossible for this dataset:

* Latitude 10.81° N is in **Somalia / the Gulf of Aden**.
* Latitude 85.45° N is in the **Arctic Ocean**, several hundred km from the nearest land.

West Eurasian archaeological sites of the Neolithic and Bronze Age should fall roughly between 35°
and 62° N. Something is wrong with some of the rows, and plotting it took five seconds.

</details>

### 📝 TODO: Find the bad rows and work out what happened to them

Pull out the rows whose latitude is outside a plausible range for this dataset — say below 30 or
above 70 — and look at them together with their population labels.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

* Combine conditions with `|` (or) and `&` (and), each in its own brackets:
  `df[(df["a"] < 1) | (df["a"] > 9)]`
* Show the columns that will help you diagnose it: `IND`, `POP`, `Latitude`, `Longitude`.
* Then ask yourself: if a `Northern_LNBA` (northern European Late Neolithic/Bronze Age) individual is
  recorded at latitude 13.6, longitude 55.4 — what is at latitude 55.4, longitude 13.6?

</details>

In [ ]:
suspect = ...                       # <- your code: rows with implausible latitude

print(suspect[["IND", "POP", "ModAnc", "Latitude", "Longitude"]].to_string(index=False))

In [ ]:
check_todo(suspect=suspect)
check(len(suspect) == 8, "There are 8 such rows.")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
suspect = geo[(geo["Latitude"] < 30) | (geo["Latitude"] > 70)]

print(suspect[["IND", "POP", "ModAnc", "Latitude", "Longitude"]].to_string(index=False))
```

</details>

### 📝 TODO: Fix them, then do the correlation

Make a corrected copy of `geo` with those eight rows' latitude and longitude exchanged, then compute
the correlation of `Latitude` and `Longitude` with `PC1` and `PC2` — before and after the fix, so
you can see what the error was doing to your conclusion.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

```python
geo_fixed = geo.copy()
bad = (geo["Latitude"] < 30) | (geo["Latitude"] > 70)
# .loc[rows, columns] = values; .values avoids pandas trying to align the column names
geo_fixed.loc[bad, ["Latitude", "Longitude"]] = geo.loc[bad, ["Longitude", "Latitude"]].values
```

`stats.pearsonr(x, y)` returns an object with `.statistic` (the coefficient *r*) and `.pvalue`.

</details>

In [ ]:
geo_fixed = ...                     # <- your code: a copy of geo with the 8 rows corrected

...                                 # <- your code: r and p for PC1/PC2 against Latitude/Longitude,
                                    #    for geo (raw) and geo_fixed

In [ ]:
check_todo(geo_fixed=geo_fixed)
check(geo_fixed["Latitude"].max() < 70 and geo_fixed["Latitude"].min() > 30,
      "After the fix all latitudes should be between 40 and 62.")
check(abs(geo_fixed["Longitude"].max() - 90.78) < 0.01,
      "The largest longitude should still be 90.78 (the Afanasievo samples in the Altai).")

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
geo_fixed = geo.copy()
bad = (geo["Latitude"] < 30) | (geo["Latitude"] > 70)
geo_fixed.loc[bad, ["Latitude", "Longitude"]] = geo.loc[bad, ["Longitude", "Latitude"]].values

rows = []
for label, table in [("raw", geo), ("fixed", geo_fixed)]:
    for pc in ["PC1", "PC2"]:
        for coord in ["Latitude", "Longitude"]:
            result = stats.pearsonr(table[pc], table[coord])
            rows.append({"data": label, "PC": pc, "coordinate": coord,
                         "r": result.statistic, "p": result.pvalue})

print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
```

</details>

### ❓ Question: Report and interpret the correlations


### 8.3 Pseudo-replication: how many independent points do you really have?

In [ ]:
print("individuals:", len(geo_fixed))
print("distinct sampling locations:", len(geo_fixed.drop_duplicates(["Latitude", "Longitude"])))

191 individuals, but only **89 distinct coordinate pairs**. Several individuals share a site exactly
— they were excavated together, often from the same burial, and are frequently from the same
culture and sometimes literally related. A correlation test assumes its data points are
**independent**; these are not. The *n* that goes into the p-value is inflated, so the p-values above
are too small.

A quick robustness check: collapse to one point per site (averaging the PCs of the individuals
found there) and redo the correlation with n = 89.

In [ ]:
by_site = (geo_fixed.groupby(["Latitude", "Longitude"])[["PC1", "PC2"]]
           .mean().reset_index())

rows = []
for pc in ["PC1", "PC2"]:
    for coord in ["Latitude", "Longitude"]:
        ind = stats.pearsonr(geo_fixed[pc], geo_fixed[coord])
        site = stats.pearsonr(by_site[pc], by_site[coord])
        rows.append({"PC": pc, "coordinate": coord,
                     "r (n=191 individuals)": ind.statistic, "p": ind.pvalue,
                     "r (n=89 sites)": site.statistic, "p ": site.pvalue})

print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

### ❓ Question: Which conclusions survive?


## 9. The East Eurasian dataset

This is a **separate PCA** with its own axes, computed from modern individuals from Russia and
Siberia, with four ancient groups projected on. `PC1` here has nothing to do with `PC1` in
`dataWE.txt`.

### 9.1 📝 Hand-in question 7: PC1 vs PC2, coloured by `POP`

Load `dataEE.txt` and plot the first two PCs, one colour per `POP`. Look at the populations and
their sizes before you interpret anything.

<details>
<summary><b>💡 Hint (click to open)</b></summary>

Same as `dataWE.txt`: `pd.read_csv(data_dir / "dataEE.txt", sep="\t")`, then `plot_pcs(...,
colour="POP")`.

</details>

In [ ]:
dataEE = ...                        # <- your code: read dataEE.txt

print("rows, columns:", dataEE.shape)
print(dataEE.groupby(["ModAnc", "POP"]).size().to_string())

In [ ]:
check_todo(dataEE=dataEE)
check(dataEE.shape == (296, 13), "dataEE should have 296 rows and 13 columns.")

In [ ]:
...                                 # <- your code: PC1 vs PC2, coloured by POP
plt.show()

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
dataEE = pd.read_csv(data_dir / "dataEE.txt", sep="\t")

print("rows, columns:", dataEE.shape)
print(dataEE.groupby(["ModAnc", "POP"]).size().to_string())

plot_pcs(dataEE, x="PC1", y="PC2", colour="POP",
         title="East Eurasian PCA: PC1 vs PC2")
plt.show()
```

</details>

### 9.2 Note

**The populations are very unequal, and one label is doing a lot of work.** `Siberians` is 225 of
the 296 individuals — one label covering many distinct peoples across a landmass the size of Europe
(Yakuts, Evenks, Tuvinians, Nganasans, Selkups and more, in the original data). That is why its
cloud is so much more spread out than any other group's: it is not one population.

In [ ]:
print(dataEE.groupby("POP")[["PC1", "PC2", "PC3"]].agg(["mean", "std"]).round(4).to_string())

One of the four projected groups is misleadingly named. The four ancient groups are
Russia_Bering, USA_Aleut, USA_Kuskokwim and Russia_Baikal. The first three are Arctic or
sub-Arctic maritime populations — the Bering Strait coast, the Aleutian Islands and southwestern
Alaska respectively. Russia_Baikal is not: Lake Baikal is in continental southern Siberia, far
from the Arctic coast, and these are ancient inland Siberians.

The data agree. In PC1–PC3 space Russia_Baikal sits 0.026 from the Siberians centroid but
0.071–0.096 from each of the three Arctic groups, and all ten of its individuals fall inside the
central 95% of the Siberian cloud on all three axes. Expect it not to behave like the other three
in the plots below.

### ❓ Question: Interpret PC1 and PC2

Which populations cluster, which separate, is it what geography predicts?


### 9.3 📝 Hand-in question 8: PC3 vs PC4

Plot PC3 against PC4 for `dataEE` and compare with PC1–PC2. How has the relationship between
populations changed?

<details>
<summary><b>💡 Hint (click to open)</b></summary>

Side by side again, as in Section 7.1.

</details>

In [ ]:
...                                 # <- your code: PC1-PC2 and PC3-PC4 of dataEE, side by side
plt.show()

<details>
<summary><b>✅ Reveal solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_pcs(dataEE, x="PC1", y="PC2", colour="POP", ax=axes[0], legend=False, title="PC1 vs PC2")
plot_pcs(dataEE, x="PC3", y="PC4", colour="POP", ax=axes[1], title="PC3 vs PC4")
plt.tight_layout()
plt.show()

print(dataEE.groupby("POP")[["PC3", "PC4"]].mean().round(4).to_string())
```

</details>

Let's run the same $\eta^2$ scan as in Section 7, since we now have the tool.

In [ ]:
separation_ee = pd.DataFrame({
    "eta_squared": [eta_squared(dataEE, pc, "POP") for pc in PCS],
    "p_value": [stats.f_oneway(*[rows[pc].values for _, rows in dataEE.groupby("POP")]).pvalue
                for pc in PCS],
}, index=PCS)

print(separation_ee.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colours = ["tab:green" if p < 0.01 else "tab:grey" for p in separation_ee["p_value"]]
ax.bar(separation_ee.index, separation_ee["eta_squared"], color=colours)
ax.set_ylabel("$\\eta^2$")
ax.set_title("East Eurasian data: how well does each PC separate the 6 populations?\n"
             "(green = p < 0.01)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### ❓ Question: How has the relationship changed on PC3 and PC4?


## 10. The general questions

### 📝 Hand-in question 9: which aDNA artefacts make ancient samples unsuitable for computing PCs?

Section 2.2 has the material. In your own words: list the artefacts, then say why that *kind* of
error is specifically a problem for a PCA.

### 📝 Hand-in question 10: what else can you analyse with PCA?

PCA is not a genetics method. Where else could it be used, in biology and outside it?
